# OSM Download from Polygon
Download OSM layers (roads, buildings, amenities, land use, green space) for a given polygon boundary.

## 1. Install & Import

In [ ]:
# !pip install osmnx geopandas shapely

In [ ]:
import re
import traceback
from pathlib import Path

import geopandas as gpd
import osmnx as ox

## 2. Settings — Edit Here

In [ ]:
# ── Input shapefile ────────────────────────────────────────────────────────
FUA_SHP      = r"D:\000_SCI\10_Compact_city\3_FUA_reference\GHS_FUA_cities_subset_clean.shp"
CITY_COL     = "eFUA_name"          # column that holds the city name

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = r"D:\000_SCI\10_Compact_city\OSM_data"

# ── Road network type ──────────────────────────────────────────────────────
# Options: "drive" | "walk" | "bike" | "all"
NETWORK_TYPE = "drive"

## 3. Load Shapefile

In [ ]:
fua = gpd.read_file(FUA_SHP)

# Reproject to EPSG:4326 (required by osmnx)
if fua.crs is None or fua.crs.to_epsg() != 4326:
    fua = fua.to_crs(epsg=4326)

print(f"Loaded {len(fua)} cities")
print(f"Columns: {list(fua.columns)}")
fua[[CITY_COL, 'geometry']].head()

## 4. Helper — Clean Before Saving

`ox.features_from_polygon` returns a MultiIndex `(element_type, osmid)`.  
When pyogrio writes this to a GeoPackage it tries to add a field called **`FID`**, which is reserved by the GPKG format and causes a `FieldError`.  
The helper below fixes that before every `.to_file()` call.

In [ ]:
import fiona

REQUIRED_LAYERS = {'roads', 'buildings', 'amenities', 'landuse', 'greenspace'}

def is_complete(gpkg_path):
    """Return True only if all 5 layers exist in the GeoPackage."""
    try:
        return REQUIRED_LAYERS.issubset(set(fiona.listlayers(str(gpkg_path))))
    except Exception:
        return False


def _clean_for_save(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Prepare an osmnx GeoDataFrame for writing to GeoPackage.

    1. Reset MultiIndex (element_type, osmid) → plain columns.
    2. Lowercase all column names  ← fixes SQLite case-insensitive conflicts
       e.g. name_zh_hant vs name_zh_Hant both become name_zh_hant.
    3. Sanitize: replace any char outside [a-z0-9_] with '_'.
    4. Rename 'fid' → 'osm_fid'  (FID is reserved by GPKG/OGR).
    5. Deduplicate column names (keep first occurrence).
    """
    import re

    gdf = gdf.reset_index()

    seen = set()
    new_cols = []
    for col in gdf.columns:
        if str(col) == 'geometry':
            new_cols.append('geometry')
            seen.add('geometry')
            continue
        # lowercase → sanitize
        c = re.sub(r"[^a-z0-9_]", "_", str(col).lower()).strip("_") or "col"
        if c[0].isdigit():
            c = "f_" + c
        if c == "fid":
            c = "osm_fid"
        # make unique
        base, n = c, 1
        while c in seen:
            c = f"{base[:60]}_{n}"
            n += 1
        seen.add(c)
        new_cols.append(c)

    gdf.columns = new_cols
    return gdf

## 5. Download Functions + Batch Loop

In [ ]:
# ── Download functions ─────────────────────────────────────────────────────

def download_roads(polygon, output_path, network_type=NETWORK_TYPE):
    """Road network edges."""
    G = ox.graph_from_polygon(polygon, network_type=network_type)
    _, edges = ox.graph_to_gdfs(G)
    edges = _clean_for_save(edges)
    edges.to_file(output_path, layer="roads", driver="GPKG")
    print(f"  roads      : {len(edges):,} segments")


def download_buildings(polygon, output_path):
    """Building footprints clipped to the polygon."""
    gdf = ox.features_from_polygon(polygon, tags={"building": True})
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = gdf.clip(polygon)
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="buildings", driver="GPKG")
    print(f"  buildings  : {len(gdf):,} footprints")


def download_amenities(polygon, output_path):
    """All OSM amenity features (schools, hospitals, restaurants, etc.)."""
    gdf = ox.features_from_polygon(polygon, tags={"amenity": True})
    gdf = gdf.clip(polygon)
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="amenities", driver="GPKG")
    print(f"  amenities  : {len(gdf):,} features")


def download_landuse(polygon, output_path):
    """Land-use polygons."""
    gdf = ox.features_from_polygon(polygon, tags={"landuse": True})
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="landuse", driver="GPKG")
    print(f"  landuse    : {len(gdf):,} polygons")


def download_greenspace(polygon, output_path):
    """Parks, forests, and green areas."""
    tags = {
        "leisure": ["park", "garden", "nature_reserve", "recreation_ground"],
        "landuse": ["forest", "grass", "meadow", "orchard", "village_green"],
        "natural": ["wood", "scrub", "heath", "grassland"],
    }
    gdf = ox.features_from_polygon(polygon, tags=tags)
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="greenspace", driver="GPKG")
    print(f"  greenspace : {len(gdf):,} polygons")


# ── Batch loop ─────────────────────────────────────────────────────────────

output_root = Path(OUTPUT_ROOT)
total, failed = len(fua), []

for i, row in fua.iterrows():
    city_name = str(row[CITY_COL])
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", city_name).strip()
    city_dir  = output_root / safe_name
    gpkg      = city_dir / f"{safe_name}_osm.gpkg"

    print(f"[{int(i)+1}/{total}] {city_name}")

    if gpkg.exists():
        if is_complete(gpkg):
            print("  Complete — skipping.")
            continue
        else:
            # Previous run failed mid-way; delete partial file and retry
            print("  Incomplete file found — deleting and redownloading...")
            gpkg.unlink()

    polygon = row.geometry
    if polygon is None or polygon.is_empty:
        print("  Empty geometry — skipping.")
        continue

    city_dir.mkdir(parents=True, exist_ok=True)

    try:
        download_roads(polygon,      gpkg)
        download_buildings(polygon,  gpkg)
        download_amenities(polygon,  gpkg)
        download_landuse(polygon,    gpkg)
        download_greenspace(polygon, gpkg)
        print(f"  Saved → {gpkg}")
    except Exception:
        print(f"  FAILED:\n{traceback.format_exc()}")
        failed.append(city_name)

print("\n" + "="*50)
print(f"Done: {total - len(failed)}/{total} cities succeeded.")
if failed:
    print("Failed:", failed)

In [ ]:
SINGLE_CITY = "Seoul"   # ← change to any city name in the shapefile

row      = fua[fua[CITY_COL] == SINGLE_CITY].iloc[0]
polygon  = row.geometry
city_dir = Path(OUTPUT_ROOT) / SINGLE_CITY
gpkg     = city_dir / f"{SINGLE_CITY}_osm.gpkg"
city_dir.mkdir(parents=True, exist_ok=True)

download_roads(polygon,      gpkg)
download_buildings(polygon,  gpkg)
download_amenities(polygon,  gpkg)
download_landuse(polygon,    gpkg)
download_greenspace(polygon, gpkg)

print("Saved →", gpkg)

## 8. Preview — Amenities with Polygon Boundary

## 7. Preview — Amenities + Boundary for All Downloaded Cities

In [ ]:
import matplotlib.pyplot as plt

output_root = Path(OUTPUT_ROOT)
save_path   = output_root / "all_cities_amenities.png"

# Collect cities that have already been downloaded
downloaded = []
for _, row in fua.iterrows():
    city_name = str(row[CITY_COL])
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", city_name).strip()
    gpkg      = output_root / safe_name / f"{safe_name}_osm.gpkg"
    if gpkg.exists():
        downloaded.append((city_name, safe_name, gpkg, row.geometry))

print(f"Found {len(downloaded)} downloaded cities: {[c[0] for c in downloaded]}")

# One subplot per city
ncols = 3
nrows = -(-len(downloaded) // ncols)   # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten() if len(downloaded) > 1 else [axes]

for ax, (city_name, safe_name, gpkg, boundary_geom) in zip(axes, downloaded):
    try:
        amenities = gpd.read_file(gpkg, layer="amenities")
        boundary  = gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326")

        pts  = amenities[amenities.geometry.geom_type == "Point"]
        poly = amenities[amenities.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]

        boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2, zorder=3)
        if not poly.empty:
            poly.plot(ax=ax, color="orange", alpha=0.5, zorder=2)
        if not pts.empty:
            pts.plot(ax=ax, color="red", markersize=2, alpha=0.6, zorder=4)

        ax.set_title(f"{city_name}\n({len(amenities):,} amenities)", fontsize=11)
    except Exception as e:
        ax.set_title(f"{city_name}\n(error: {e})", fontsize=9)
    ax.set_axis_off()

# Hide unused subplots
for ax in axes[len(downloaded):]:
    ax.set_visible(False)

plt.suptitle("Amenities by City", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()

## 8. UML — Activity Diagram

```mermaid
flowchart TD
    START([START]) --> LOAD

    LOAD["Load FUA Shapefile
    gpd.read_file · reproject EPSG:4326"]

    LOAD --> LOOP[/"⟳ For each city in FUA"/]
    LOOP --> CHK_EXIST{"'.gpkg'\nexists?"}

    CHK_EXIST -- Yes --> SKIP[Skip city]
    SKIP -.->|next city| LOOP

    CHK_EXIST -- No --> CHK_GEOM{"Valid\ngeometry?"}
    CHK_GEOM -- No --> SKIP
    CHK_GEOM -- Yes --> MKDIR["mkdir  output_root / city_name /"]

    MKDIR --> DL_ROADS["download_roads
    graph_from_polygon → graph_to_gdfs"]

    DL_ROADS --> DL_BUILD["download_buildings
    features_from_polygon · keep polygons · clip"]

    DL_BUILD --> DL_AMEN["download_amenities
    features_from_polygon · clip"]

    DL_AMEN --> DL_LAND["download_landuse
    features_from_polygon · keep polygons"]

    DL_LAND --> DL_GREEN["download_greenspace
    features_from_polygon · keep polygons"]

    DL_GREEN --> CLEAN

    subgraph CLEAN["_clean_for_save()"]
        direction LR
        C1[reset_index] --> C2[sanitize col names]
        C2 --> C3["rename FID → osm_fid"]
        C3 --> C4[drop duplicates]
    end

    CLEAN --> SAVE["to_file · layer · GPKG
    roads · buildings · amenities · landuse · greenspace"]

    SAVE --> CHK_ERR{"Exception\nraised?"}
    CHK_ERR -- Yes --> FAIL["Log FAILED
    append to failed list"]
    FAIL -.->|next city| LOOP

    CHK_ERR -- No --> DONE["✓ Saved → city_osm.gpkg"]
    DONE -.->|next city| LOOP

    LOOP --> SUMMARY["Print summary
    N / total succeeded · list failed cities"]
    SUMMARY --> PREVIEW["Preview Plot
    all downloaded cities · amenities + boundary → PNG"]
    PREVIEW --> END([END])
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.patches import Polygon as MplPolygon

fig, ax = plt.subplots(figsize=(12, 22))
ax.set_xlim(0, 12)
ax.set_ylim(0, 22)
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')
ax.axis('off')

# ── Colours ────────────────────────────────────────────────────────────────
BG = {'io': '#D6EAF8', 'proc': '#D5F5E3', 'dec': '#FDEBD0',
      'help': '#FADBD8', 'term': '#D7BDE2', 'skip': '#F2F3F4', 'fail': '#FADBD8'}
EC = '#2C3E50'

# ── Drawing helpers ────────────────────────────────────────────────────────
def rbox(cx, cy, w, h, txt, bg, fs=8.3):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                 boxstyle='round,pad=0.07', facecolor=bg, edgecolor=EC, lw=1.3, zorder=3))
    ax.text(cx, cy, txt, ha='center', va='center', fontsize=fs,
            multialignment='center', zorder=4)

def diam(cx, cy, hw, hh, txt, bg=BG['dec'], fs=8):
    pts = [(cx, cy+hh), (cx+hw, cy), (cx, cy-hh), (cx-hw, cy)]
    ax.add_patch(MplPolygon(pts, closed=True, facecolor=bg, edgecolor=EC, lw=1.3, zorder=3))
    ax.text(cx, cy, txt, ha='center', va='center', fontsize=fs,
            multialignment='center', zorder=4)

def term(cx, cy, txt):
    ax.add_patch(mpatches.Ellipse((cx, cy), 2.8, 0.65,
                 facecolor=BG['term'], edgecolor=EC, lw=1.8, zorder=3))
    ax.text(cx, cy, txt, ha='center', va='center', fontsize=10,
            fontweight='bold', zorder=4)

def varr(x, y1, y2, lbl='', lx=0.15):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
                arrowprops=dict(arrowstyle='->', color=EC, lw=1.3), zorder=5)
    if lbl:
        ax.text(x+lx, (y1+y2)/2, lbl, fontsize=7.5, color='#555', va='center')

def harr(x1, y, x2, lbl=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color=EC, lw=1.3), zorder=5)
    if lbl:
        ax.text((x1+x2)/2, y+0.13, lbl, fontsize=7.5, color='#555', ha='center')

def line(xs, ys):
    ax.plot(xs, ys, color=EC, lw=1.3, zorder=5)

# ── Layout ─────────────────────────────────────────────────────────────────
X, W, H = 5.5, 5.2, 0.65      # centre-x, box-width, box-height
SX = 9.8                        # side-column x (skip / fail)

Y = dict(
    start=21.2, load=19.9, loop=18.5,
    exist=17.1, geom=15.6, mkdir=14.3,
    roads=13.1, build=12.0, amen=10.9,
    land=9.8,   green=8.7,  clean=7.4,
    save=6.2,   err=4.9,    summary=3.6,
    end=2.3,
)

# ── Nodes ──────────────────────────────────────────────────────────────────
term(X, Y['start'], 'START')
rbox(X, Y['load'],  W, H,   'Load FUA Shapefile\ngpd.read_file()  →  reproject EPSG:4326', BG['io'])
rbox(X, Y['loop'],  W, 0.6, '⟳  For each city in FUA', '#EBF5FB')
diam(X, Y['exist'], 1.9, 0.55, '.gpkg\nexists?')
diam(X, Y['geom'],  1.9, 0.55, 'Valid\ngeometry?')
rbox(X, Y['mkdir'], W, 0.6, 'mkdir  output_root / city_name /', BG['proc'])
rbox(X, Y['roads'], W, H,   'download_roads\ngraph_from_polygon  →  graph_to_gdfs', BG['proc'])
rbox(X, Y['build'], W, H,   'download_buildings\nfeatures_from_polygon  +  clip(polygon)', BG['proc'])
rbox(X, Y['amen'],  W, H,   'download_amenities\nfeatures_from_polygon  +  clip(polygon)', BG['proc'])
rbox(X, Y['land'],  W, H,   'download_landuse\nfeatures_from_polygon  (polygons only)', BG['proc'])
rbox(X, Y['green'], W, H,   'download_greenspace\nfeatures_from_polygon  (polygons only)', BG['proc'])
rbox(X, Y['clean'], W, 0.85,
     '_clean_for_save()\n① reset_index   ② sanitize col names   ③ rename FID→osm_fid   ④ drop dups',
     BG['help'])
rbox(X, Y['save'],  W, H,
     'gdf.to_file( city_osm.gpkg,  layer,  GPKG )\nroads · buildings · amenities · landuse · greenspace',
     BG['io'])
diam(X, Y['err'],   1.9, 0.55, 'Exception\nraised?')
rbox(X, Y['summary'], W, H, 'Print summary  ·  N/total succeeded  ·  list failed cities', BG['io'])
term(X, Y['end'], 'END')

# Side column
rbox(SX, Y['exist'], 2.4, 0.55, '⏭  Skip city', BG['skip'])
rbox(SX, Y['err'],   2.4, 0.55, '✕  Log FAILED', BG['fail'])

# ── Main arrows ────────────────────────────────────────────────────────────
varr(X, Y['start']-0.33, Y['load']+H/2)
varr(X, Y['load']-H/2,   Y['loop']+0.3)
varr(X, Y['loop']-0.3,   Y['exist']+0.55)
varr(X, Y['exist']-0.55, Y['geom']+0.55,  lbl='No')
varr(X, Y['geom']-0.55,  Y['mkdir']+H/2,  lbl='Yes')
varr(X, Y['mkdir']-H/2,  Y['roads']+H/2)
varr(X, Y['roads']-H/2,  Y['build']+H/2)
varr(X, Y['build']-H/2,  Y['amen']+H/2)
varr(X, Y['amen']-H/2,   Y['land']+H/2)
varr(X, Y['land']-H/2,   Y['green']+H/2)
varr(X, Y['green']-H/2,  Y['clean']+0.43)
varr(X, Y['clean']-0.43, Y['save']+H/2)
varr(X, Y['save']-H/2,   Y['err']+0.55)
varr(X, Y['err']-0.55,   Y['summary']+H/2, lbl='No')
varr(X, Y['summary']-H/2, Y['end']+0.33)

# exists? → skip box
harr(X+1.9, Y['exist'], SX-1.2, lbl='Yes')
# geom invalid → same skip box (line down then arrow up)
line([X+1.9, SX-1.2], [Y['geom'], Y['geom']])
line([SX-1.2, SX-1.2], [Y['geom'], Y['exist']-0.28])
ax.annotate('', xy=(SX-1.2, Y['exist']-0.28), xytext=(SX-1.2, Y['geom']),
            arrowprops=dict(arrowstyle='->', color=EC, lw=1.3), zorder=5)
ax.text(X+2.05, Y['geom']+0.12, 'No', fontsize=7.5, color='#555')

# skip → back to loop
line([SX, SX], [Y['exist']+0.28, Y['loop']+0.1])
line([SX, X+W/2], [Y['loop']+0.1, Y['loop']+0.1])

# exception → fail
harr(X+1.9, Y['err'], SX-1.2, lbl='Yes')
# fail → back to loop
line([SX, SX], [Y['err']+0.28, Y['loop']-0.1])
line([SX, X+W/2], [Y['loop']-0.1, Y['loop']-0.1])
ax.annotate('', xy=(X+W/2, Y['loop']-0.1), xytext=(SX, Y['loop']-0.1),
            arrowprops=dict(arrowstyle='->', color=EC, lw=1.3), zorder=5)

# ── Legend ─────────────────────────────────────────────────────────────────
legend = [
    mpatches.Patch(facecolor=BG['term'], edgecolor=EC, label='Start / End'),
    mpatches.Patch(facecolor=BG['io'],   edgecolor=EC, label='Input / Output'),
    mpatches.Patch(facecolor=BG['proc'], edgecolor=EC, label='Process'),
    mpatches.Patch(facecolor=BG['dec'],  edgecolor=EC, label='Decision'),
    mpatches.Patch(facecolor=BG['help'], edgecolor=EC, label='Helper / Error'),
]
ax.legend(handles=legend, loc='lower left', fontsize=8.5,
          framealpha=0.95, bbox_to_anchor=(0.0, 0.0))

ax.set_title('OSM Polygon Downloader — UML Activity Diagram',
             fontsize=14, fontweight='bold', pad=14)

plt.tight_layout()
uml_path = Path(OUTPUT_ROOT) / 'uml_activity_diagram.png'
plt.savefig(uml_path, dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
print(f'Saved → {uml_path}')
plt.show()